# Persistent cache: setup speed and correctness

A fixed random problem compares analytical (cold) setup with a warm disk cache. CUDA-full is timed; CPU-static/oneMKL checks bitwise-identical cached and uncached fields.

In [ ]:
from contextlib import contextmanager
import ctypes
import os
import sys
import time

import cdfmm
import numpy as np


@contextmanager
def suppress_initialization_output():
    """Hide the native initialization report for this compact notebook."""
    sys.stdout.flush()
    libc = ctypes.CDLL(None)
    libc.fflush(None)
    saved_stdout = os.dup(1)
    try:
        with open(os.devnull, "w") as sink:
            os.dup2(sink.fileno(), 1)
            yield
            libc.fflush(None)
    finally:
        os.dup2(saved_stdout, 1)
        os.close(saved_stdout)

## Problem

In [ ]:
N = 50000
ORDER = 6
DEPTH = 4
SEED = 42
PRECISION = cdfmm.StaticPrecision.FLOAT32

rng = np.random.default_rng(SEED)
positions = rng.uniform(-0.5, 0.5, size=(N, 3))
moments = rng.normal(size=(N, 3))
fixed_identity = np.arange(N, dtype=int).tolist()

In [ ]:
def make_options(backend, cache_enabled):
    options = cdfmm.UniformFmmOptions()
    options.enable_cache = cache_enabled
    options.expansion_basis = cdfmm.ExpansionBasis.Spherical
    options.expansion_order = ORDER
    options.precision = PRECISION
    options.tree.max_level = DEPTH
    options.backend = backend
    options.static_matrix_backend = cdfmm.StaticMatrixBackend.ONE_MKL
    options.fixed_target_source_indices = fixed_identity
    return options


def construct(backend, cache_enabled):
    options = make_options(backend, cache_enabled)
    started = time.perf_counter()
    with suppress_initialization_output():
        plan = cdfmm.UniformFmm(positions, positions, options)
    elapsed = time.perf_counter() - started
    return plan, elapsed, dict(plan.static_plan_statistics)


def geometry_setup_time(statistics):
    names = (
        "p2m_construction_seconds",
        "m2m_construction_seconds",
        "m2l_construction_seconds",
        "l2l_construction_seconds",
        "l2p_construction_seconds",
        "p2p_construction_seconds",
    )
    return sum(statistics[name] for name in names)


def speedup(cold, warm):
    return cold / warm if warm > 0.0 else float("inf")

## Run comparison

In [ ]:
# CUDA setup timing. The seed construction guarantees a warm cache even
# when this is the first notebook run.
cuda_cold, cuda_cold_wall, cuda_cold_stats = construct(
    cdfmm.ExecutionBackend.CUDA_FULL, cache_enabled=False
)
cache_seed, _, _ = construct(
    cdfmm.ExecutionBackend.CUDA_FULL, cache_enabled=True
)
cuda_warm, cuda_warm_wall, cuda_warm_stats = construct(
    cdfmm.ExecutionBackend.CUDA_FULL, cache_enabled=True
)

assert cuda_warm_stats["universal_cache_hit"]
assert cuda_warm_stats["geometry_cache_hit"]

# CPU/oneMKL is used here because separate CUDA evaluations can differ in
# the last FP32 bit independently of cache use.
cpu_cold, cpu_cold_wall, cpu_cold_stats = construct(
    cdfmm.ExecutionBackend.CPU_STATIC, cache_enabled=False
)
cpu_warm, cpu_warm_wall, cpu_warm_stats = construct(
    cdfmm.ExecutionBackend.CPU_STATIC, cache_enabled=True
)
cold_field = cpu_cold.evaluate(moments, output="field")["H"]
warm_field = cpu_warm.evaluate(moments, output="field")["H"]
same_result = np.array_equal(cold_field, warm_field)

In [ ]:
cold_universal = cuda_cold_stats["universal_operator_build_seconds"]
warm_universal = cuda_warm_stats["universal_cache_load_seconds"]
cold_geometry = geometry_setup_time(cuda_cold_stats)
warm_geometry = cuda_warm_stats["geometry_cache_load_seconds"]
cold_total = cuda_cold_stats["total_setup_seconds"]
warm_total = cuda_warm_stats["total_setup_seconds"]

print(f"N={N:,}, order={ORDER}, depth={DEPTH}")
print()
print(f"{'Setup':<22} {'Cold (s)':>12} {'Warm (s)':>12} {'Speedup':>10}")
print("-" * 60)
for label, cold, warm in (
    ("Universal static", cold_universal, warm_universal),
    ("Geometry dependent", cold_geometry, warm_geometry),
    ("Total initialization", cold_total, warm_total),
):
    print(f"{label:<22} {cold:12.6f} {warm:12.6f} {speedup(cold, warm):9.2f}x")
print()
print(f"Same result: {same_result}")